# ch11 练习 —— 变换与函数应用

数据：`data/device_defects.csv`（270 行 x 13 列）

**每题都有 `assert` 验收块**。跑通即正确，跑不通的报错信息本身就说清了哪里错。
**不要翻答案版。**

| 题号 | 考什么 |
|---|---|
| TODO(1) | `map(dict)` 等级映射 |
| TODO(2) | `map` 未命中的静默后果 |
| TODO(3) | `replace(dict)` 与 `map` 的差别 |
| TODO(4) | `df.apply(axis=0)` 逐列统计 |
| TODO(5) | `df.apply(axis=1)` 逐行计算 |
| TODO(6) | 向量化替代 `axis=1` |
| TODO(7) | `gb.apply` 返回 `Series` 的形状 |
| TODO(8) | `gb.apply` 返回 `DataFrame` 的**多级索引** |
| TODO(9) | `include_groups` 在 3.0 的真实行为 |
| TODO(10) | `DataFrame.map` 取代 `applymap` |
| 综合 ① | `transform` 造组内特征（对照 `agg`） |
| 综合 ② | `pipe` 串一条清洗流水线 |

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

DATA = Path("data")
dev = pd.read_csv(DATA / "device_defects.csv")

GROUP_KEY = "设备类型"
VALUE_COL = "负荷值"
NUM_COLS = ["负荷值", "温度", "湿度"]

LEVEL_SCORE = {"一般": 1, "严重": 2, "危急": 3}

# 脚手架函数：写在 @@todo 之外，练习版原样保留
def add_relay_flag(df, col=VALUE_COL, thr=500):
    """给 DataFrame 打一个高负荷标记列。"""
    out = df.copy()
    out["高负荷"] = out[col] > thr
    return out


def row_range(row):
    """一行内数值列的极差（nanmax - nanmin）。"""
    values = row[NUM_COLS].to_numpy(dtype="float64")
    return float(np.nanmax(values) - np.nanmin(values))


def group_span(frame):
    """一个分组内部的负荷值极差。"""
    return float(frame[VALUE_COL].max() - frame[VALUE_COL].min())


def drop_sentinel(df, lo=0, hi=1000, col=VALUE_COL):
    """丢掉负荷值落在 [lo, hi] 之外的哨兵行。"""
    return df[df[col].between(lo, hi)]


def add_ratio(df, col=VALUE_COL, key=GROUP_KEY):
    """加一列：该记录在本分组内的负荷占比。"""
    out = df.copy()
    out["组内占比"] = out[col] / out.groupby(key)[col].transform("sum")
    return out


def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。

    返回 ("ok", 结果) 或 ("err", 异常类型名)。
    """
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


print("pandas", pd.__version__)
print("dev", dev.shape)

In [ ]:
gb = dev.groupby(GROUP_KEY)
big = pd.concat([dev] * 400, ignore_index=True)
print("gb:", type(gb).__name__, "| big:", big.shape)

---

## 基础题

In [ ]:
score = dev["缺陷等级"].map(LEVEL_SCORE)
# ---- 验收 ----
assert score.isna().sum() == 0, "三个等级都在映射表里，不该有未命中"
assert score.value_counts().to_dict() == {1: 164, 2: 75, 3: 31}
assert str(score.dtype) == "int64", f"dtype 应为 int64，实际 {score.dtype}"
assert len(score) == len(dev), "元素级映射保持长度不变"
print("TODO(1) 通过 -> 分值分布", score.value_counts().to_dict())

In [ ]:
partial = dev["缺陷等级"].map({"一般": 1})
n_miss = int(partial.isna().sum())
# ---- 验收 ----
assert n_miss == 106, f"未命中应为 106（270 - 一般 164），实际 {n_miss}"
assert n_miss == len(dev) - int((dev["缺陷等级"] == "一般").sum()), "未命中数 = 总行数 - 命中行数"
# 对照：同样不完整的 replace 不会产生 NaN
partial_r = dev["缺陷等级"].replace({"一般": 1})
assert int(partial_r.isna().sum()) == 0, "replace 未命中会保留原值，不产生 NaN"
assert partial_r.nunique() == 3, "replace 后仍有 1 / 严重 / 危急 三种取值"
print("TODO(2) 通过 -> map 未命中产生", n_miss, "个 NaN；replace 未命中产生 0 个")

In [ ]:
#         先去空白，再合并，结果存入 defect_merged
defect_merged = dev["缺陷类型"].str.strip().replace({
    "渗油": "渗漏油",
    "漏油": "渗漏油",
    "异物搭挂": "异物",
})
# ---- 验收 ----
assert defect_merged.nunique() == 6, f"合并后应为 6 类，实际 {defect_merged.nunique()}"
assert int(defect_merged.isna().sum()) == 0, "replace 不该产生缺失（变出 NaN 说明用错成 map 了）"
assert defect_merged.value_counts().to_dict() == {
    "渗漏油": 61, "发热": 51, "放电痕迹": 44, "异物": 40, "破损": 38, "锈蚀": 36,
}
print("TODO(3) 通过 -> 合并为", defect_merged.nunique(), "类:", defect_merged.value_counts().to_dict())

---

## 进阶题

In [ ]:
col_range = dev[NUM_COLS].apply(lambda s: s.max() - s.min())
# ---- 验收 ----
assert isinstance(col_range, pd.Series), f"axis=0 应返回 Series，实际 {type(col_range).__name__}"
assert list(col_range.index) == NUM_COLS, "index 应是列名"
assert abs(col_range["负荷值"] - 10000.0) < 1e-6, f"负荷值极差应为 10000.0（9999-(-1)），实际 {col_range['负荷值']}"
assert abs(col_range["温度"] - 45.3) < 1e-6, f"温度极差应为 45.3，实际 {col_range['温度']}"
assert abs(col_range["湿度"] - 155.0) < 1e-6, f"湿度极差应为 155.0，实际 {col_range['湿度']}"
print("TODO(4) 通过 -> 逐列极差", col_range.round(3).to_dict())

In [ ]:
row_range_series = dev[NUM_COLS].apply(row_range, axis=1)
# ---- 验收 ----
assert isinstance(row_range_series, pd.Series), "axis=1 应返回 Series"
assert len(row_range_series) == 270, f"长度应等于行数，实际 {len(row_range_series)}"
assert list(row_range_series.index) == list(dev.index), "index 应是行索引"
assert np.allclose(row_range_series.head(3).round(2).tolist(), [61.16, 180.99, 34.61], atol=0.01), \
    f"前 3 行应为 [61.16, 180.99, 34.61]，实际 {row_range_series.head(3).round(2).tolist()}"
assert row_range_series.notna().all(), "极差计算不应产生缺失"
print("TODO(5) 通过 -> 前 3 行行内极差:", row_range_series.head(3).round(2).tolist())

In [ ]:
num_block = dev[NUM_COLS]
row_range_fast = num_block.max(axis=1) - num_block.min(axis=1)
# ---- 验收 ----
assert len(row_range_fast) == 270
assert bool(np.allclose(row_range_fast.to_numpy(), row_range_series.to_numpy(), equal_nan=True)), \
    "向量化结果必须与 apply(axis=1) 完全一致"
# 实测耗时对比（只打印，不做断言——计时在不同机器上会漂）
t0 = time.perf_counter()
_ = num_block.apply(row_range, axis=1)
t1 = time.perf_counter()
_ = num_block.max(axis=1) - num_block.min(axis=1)
t2 = time.perf_counter()
print(f"TODO(6) 通过 -> 结果一致；apply(axis=1) {t1 - t0:.4f}s vs 向量化 {t2 - t1:.4f}s"
      f"（快 {(t1 - t0) / max(t2 - t1, 1e-9):.0f} 倍）")

In [ ]:
r_series = gb.apply(lambda d: d[[VALUE_COL, "湿度"]].mean(), include_groups=False)
# ---- 验收 ----
assert isinstance(r_series, pd.DataFrame), f"返回 Series 时 apply 结果应是 DataFrame，实际 {type(r_series).__name__}"
assert r_series.shape == (5, 2), f"形状应为 (5, 2)，实际 {r_series.shape}"
assert r_series.index.names == [GROUP_KEY], f"index 名应为 [{GROUP_KEY}]，实际 {r_series.index.names}"
assert list(r_series.columns) == [VALUE_COL, "湿度"]
assert abs(r_series.loc["变压器", VALUE_COL] - 986.9231) < 1e-3
print("TODO(7) 通过 ->", r_series.shape)
print(r_series.round(2).to_string())

In [ ]:
r_frame = gb.apply(lambda d: d.nlargest(2, VALUE_COL), include_groups=False)
# ---- 验收 ----
assert isinstance(r_frame, pd.DataFrame), "应返回 DataFrame"
assert r_frame.shape == (10, 12), f"形状应为 (10, 12)（5 组 x 2 行，12 列＝原 13 列去掉分组键），实际 {r_frame.shape}"
assert r_frame.index.nlevels == 2, f"索引应有 2 级，实际 {r_frame.index.nlevels}"
assert r_frame.index.names == [GROUP_KEY, None], f"索引名应为 [{GROUP_KEY}, None]，实际 {r_frame.index.names}"
assert GROUP_KEY not in r_frame.columns, "分组键在索引上，不在列里"
# 处理掉多余的一级原索引
flat = r_frame.reset_index(level=1, drop=True).reset_index()
assert flat.shape[0] == 10, "行数不能变"
assert flat.shape[1] == 13, f"reset_index 后应恢复成 13 列，实际 {flat.shape[1]}"
assert GROUP_KEY in flat.columns, "reset_index 后分组键回到列里"
assert flat.index.tolist() == list(range(10)), "应得到干净的 RangeIndex"
print("TODO(8) 通过 -> r_frame", r_frame.shape, "索引", r_frame.index.names)
print(flat[[GROUP_KEY, VALUE_COL, "缺陷等级"]].to_string(index=False))

In [ ]:
#         include_groups=False 的返回值存入 ok_result；
#         用脚手架里的 probe 探测 include_groups=True，结果存入 probe_true
ok_result = gb.apply(lambda d: d[VALUE_COL].mean(), include_groups=False)
probe_true = probe(
    lambda **kw: gb.apply(lambda d: d[VALUE_COL].mean(), **kw), include_groups=True
)
# ---- 验收 ----
assert ok_result is not None, "include_groups=False 应该可用"
assert len(ok_result) == 5, f"应有 5 组，实际 {len(ok_result)}"
assert probe_true[0] == "err", "include_groups=True 在 pandas 3.0 应直接报错"
assert probe_true[1] == "ValueError", f"include_groups=True 应抛 ValueError，实际 {probe_true[1]}"
# 关键行为：include_groups=False 时 func 收到的子表里没有分组键
has_key_in_apply = bool(gb.apply(lambda d: GROUP_KEY in d.columns, include_groups=False).any())
assert has_key_in_apply is False, "include_groups=False 下子表不该含分组键"
# 对照：直接 for/next 迭代 GroupBy 拿到的子表**仍然含**分组键（这一步不受 include_groups 影响）
sub = next(iter(gb))[1]
assert GROUP_KEY in sub.columns, "直接迭代 GroupBy 的子表仍保留分组键（与 apply 不同！）"
assert len(sub.columns) == 13, f"直接迭代应为 13 列，实际 {len(sub.columns)}"
r_frame_cols = gb.apply(lambda d: d.nlargest(1, VALUE_COL), include_groups=False).columns
assert len(r_frame_cols) == 12, f"apply(include_groups=False) 的子表应为 12 列，实际 {len(r_frame_cols)}"
print("TODO(9) 通过 -> include_groups=False 可用；=True 抛", probe_true[1])
print("   直接迭代列数:", len(sub.columns), "| apply(include_groups=False) 列数:", len(r_frame_cols))

In [ ]:
#         并记录 applymap 是否还存在（布尔量存入 has_applymap）
rounded = dev[NUM_COLS].map(lambda v: round(v, 1) if pd.notna(v) else v)
has_applymap = hasattr(dev, "applymap")
# ---- 验收 ----
assert has_applymap is False, "applymap 在 pandas 3.0 已被移除"
assert hasattr(dev, "map") is True, "DataFrame.map 存在"
assert rounded.shape == (270, 3), f"形状应保持 (270, 3)，实际 {rounded.shape}"
assert rounded[VALUE_COL].head(2).tolist() == [90.0, 180.1], \
    f"前 2 个负荷值应为 [90.0, 180.1]，实际 {rounded[VALUE_COL].head(2).tolist()}"
assert int(rounded[VALUE_COL].isna().sum()) == 21, "缺失必须原样保留（不能被 round 成 0）"
# 混类型表会崩
try:
    dev[["负荷值", "缺陷等级"]].map(lambda v: round(v, 1))
    raise AssertionError("混类型表上 DataFrame.map 应该抛 TypeError")
except AssertionError:
    raise
except TypeError:
    pass
print("TODO(10) 通过 -> applymap 存在:", has_applymap, "| 数值块 map 后前 2 值:",
      rounded[VALUE_COL].head(2).tolist())

---

## 综合题

### 综合 ① `transform` 造组内特征（并对照 `agg`）

用 `transform` 造三列组内特征，同时用实验证明 **`agg` 会压缩行数、`transform` 不会**。

三列特征：组均值、组内 z-score、组内百分位排名。

In [ ]:
#       agg_cmp = dev.groupby(GROUP_KEY)[VALUE_COL].mean()
feat = dev[[GROUP_KEY, VALUE_COL]].copy()
feat["组均值"] = feat.groupby(GROUP_KEY)[VALUE_COL].transform("mean")
feat["组内z"] = (feat[VALUE_COL] - feat["组均值"]) / feat.groupby(GROUP_KEY)[VALUE_COL].transform("std")
feat["组内排名"] = feat.groupby(GROUP_KEY)[VALUE_COL].rank(pct=True)
agg_cmp = dev.groupby(GROUP_KEY)[VALUE_COL].mean()
# ---- 验收 ----
assert feat.shape == (270, 5), f"feat 形状应为 (270, 5)，实际 {feat.shape}"
assert len(feat) == len(dev), "transform 保持行数"
assert len(agg_cmp) == 5, "agg 压缩成 5 行"
assert len(feat) != len(agg_cmp), "这就是 transform 与 agg 的根本区别"
assert list(feat.columns) == [GROUP_KEY, VALUE_COL, "组均值", "组内z", "组内排名"]
# 组均值：同一组内应只有一个值，且等于 agg 的结果
for grp, sub in feat.groupby(GROUP_KEY):
    assert sub["组均值"].nunique() == 1, f"{grp} 组内组均值应唯一"
    assert abs(sub["组均值"].iloc[0] - agg_cmp[grp]) < 1e-9, f"{grp} 组 transform 值与 agg 值不一致"
# z-score 与讲解一致
assert np.allclose(feat["组内z"].head(3).round(3).tolist(), [-0.179, -0.221, 0.218], atol=1e-3)
# 组内排名是百分位，落在 (0, 1]
assert feat["组内排名"].min() > 0 and feat["组内排名"].max() == 1.0
# 每组的排名均值应接近 0.5
assert abs(feat.groupby(GROUP_KEY)["组内排名"].mean().mean() - 0.5) < 0.02
print("综合① 通过 -> feat", feat.shape, "vs agg_cmp", agg_cmp.shape)
print(feat.head(3).round(3).to_string(index=False))

### 综合 ② `pipe` 串一条清洗流水线

用 `pipe` 把三步清洗串起来，全程不出现中间变量：

1. `drop_sentinel`：去掉负荷值不在 `[0, 1000]` 的行（清哨兵与负值）
2. `add_relay_flag`：打高负荷标记（阈值 500）
3. `add_ratio`：算每条记录在本设备类型内的负荷占比

这三个函数**已写在首格脚手架里**（`pipe` 考的是「串」而不是「写函数」），
你只需要把它们按顺序接到 `dev` 后面，并产出报告字典 `pipeline`。

要求最后一步的占比满足"各组之和 == 1"，这是流水线造特征正确的证明。

In [ ]:
# pipeline = {"原始行数": len(dev), "清洗后行数": len(chain),
#             "清掉行数": len(dev) - len(chain), "高负荷条数": int(chain["高负荷"].sum())}
chain = (
    dev.pipe(drop_sentinel, lo=0, hi=1000)
    .pipe(add_relay_flag, thr=500)
    .pipe(add_ratio)
)
pipeline = {
    "原始行数": len(dev),
    "清洗后行数": len(chain),
    "清掉行数": len(dev) - len(chain),
    "高负荷条数": int(chain["高负荷"].sum()),
}
# ---- 验收 ----
assert set(pipeline) == {"原始行数", "清洗后行数", "清掉行数", "高负荷条数"}
assert pipeline["原始行数"] == 270
assert pipeline["清掉行数"] == 32, (
    f"应清掉 32 行（7 个 >=1000 + 4 个负值 + 21 个缺失），实际 {pipeline['清掉行数']}"
)
assert pipeline["清洗后行数"] == 238
assert pipeline["高负荷条数"] == int((chain[VALUE_COL] > 500).sum())
assert pipeline["高负荷条数"] == 38, f"清洗后负荷值>500 的应为 38 条，实际 {pipeline['高负荷条数']}"
# 新增列
new_cols = [c for c in chain.columns if c not in dev.columns]
assert sorted(new_cols) == ["组内占比", "高负荷"], f"新增列不对：{new_cols}"
# 流水线正确性证明：各组组内占比之和 == 1
ratio_sum = chain.groupby(GROUP_KEY)["组内占比"].sum()
assert np.allclose(ratio_sum.to_numpy(), 1.0, atol=1e-9), f"各组占比之和应为 1，实际 {ratio_sum.to_dict()}"
assert len(chain) < len(dev), "清洗应该减少行数"
print("综合② 通过 -> 流水线报告：")
for key, value in pipeline.items():
    print(f"   {key:10s} = {value}")
print("   各组组内占比之和:", ratio_sum.round(6).to_dict())

> **⚠️ 这道题里藏着一个静默陷阱**：你算出的「清掉行数」是 **32**，不是 11。
>
> 32 = 7 个 `>= 1000` 的哨兵 + 4 个负值 + **21 个缺失值**。
> `Series.between(0, 1000)` 对 `NaN` 返回 `False`，所以缺失值会被**一起丢掉**，
> 而且不报错、不警告。
>
> - 如果「丢弃缺失」是你想要的 → 正好，一行搞定
> - 如果「缺失要保留、只清哨兵」→ 得写成
>   `df[df[col].isna() | df[col].between(lo, hi)]`
>
> 竞赛里这类"顺手多丢了几行"极难被发现——**直到最后对不上分组统计的样本数**。
> 记住规则：**任何基于比较的过滤（`between` / `>` / `==` / `isin`）都会静默丢掉 `NaN`。**

---

### 复盘提问

**复盘问自己**（六题都能答上来才算过）：

1. `map(dict)` 和 `replace(dict)` 在"遇到字典里没有的值"时分别怎么做？各自适合什么场景？
2. `df.apply(axis=1)` 在这份 270 行数据上比向量化慢多少？为什么 10 万行就不可接受了？
3. `gb.apply` 返回 `Series` / `DataFrame` 时，结果的形状和索引分别是什么？多出来的一级怎么处理？
4. `include_groups=True` 在 pandas 3.0 会发生什么？为什么"直接 `for` 迭代"拿到的子表**仍然含**分组键？
5. 你有一个"等级 → 分值"的需求，写出三种实现，并按速度排序。
6. 用 `between` 做过滤时，缺失值去哪了？想保留它该怎么写？

答不上来的，回 `_notes/错题本.md` 记一笔。